In [1]:
from kaldo.interfaces.tdep_io import (
    remap_force_constants, 
    parse_tdep_forceconstant,
    parse_tdep_third_forceconstant
)
from kaldo.forceconstants import ForceConstants
from kaldo.phonons import Phonons

import ase
import os
import numpy as np

2026-05-07 12:05:23.603532: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 12:05:23.669455: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-07 12:05:25.471500: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
basepath = os.path.abspath("./SW")
fc_path = os.path.join(basepath, "100K_3UC", "infile.forceconstant")
fc3_path = os.path.join(basepath, "100K_3UC", "infile.forceconstant_thirdorder")
fc4_path = os.path.join(basepath, "100K_3UC", "infile.forceconstant_fourthorder")
uc_path = os.path.join(basepath, "infile.ucposcar")
sc_path = os.path.join(basepath, "infile.ssposcar")

uc = ase.io.read(uc_path, format = "vasp")
sc = ase.io.read(sc_path, format = "vasp")
M = np.linalg.solve(np.asarray(uc.cell), np.asarray(sc.cell))
print(uc)
print(sc)
print(M)

Atoms(symbols='Si2', pbc=True, cell=[[2.715, 2.715, 0.0], [0.0, 2.715, 2.715], [2.715, 0.0, 2.715]])
Atoms(symbols='Si216', pbc=True, cell=[16.29, 16.29, 16.29])
[[ 3. -3.  3.]
 [ 3.  3. -3.]
 [-3.  3.  3.]]


In [4]:
tdep_dir = os.path.abspath("./SW/100K_3UC")
#! Parsing 4th order is way slower than it should be
fc = ForceConstants.from_folder(
        folder=str(tdep_dir), format="tdep", supercell = (3,3,3), include_fourth=False,
    )

(108, 1, 1)


In [5]:
fc.second.value.shape

(1, 2, 3, 108, 2, 3)

In [6]:
fc.third.value.shape
# (N_uc, N_sc, N_sc, 3, 3, 3) --> (2, 2*108, 2*108, 3, 3, 3)
# kaldo stores as (2, 3, 108, 2, 3, 108, 2, 3)

(2, 3, 108, 2, 3, 108, 2, 3)

In [7]:
fc.second._direct_grid.id_to_grid_index(30)

array([ 3,  0, -1])

In [8]:
fc.third.supercell

(108, 1, 1)

In [9]:
ph = Phonons(forceconstants=fc)

In [5]:
from kaldo.remap2 import remap_third_force_constants, remap_third_force_constants_integer


In [6]:
ifc3_sc = remap_third_force_constants(fc.third, sc)

Remap IFC3 pairs: 100%|██████████| 62424/62424 [00:33<00:00, 1866.04it/s]


In [27]:
np.amax(ifc3_sc.value)

np.float64(27.29781906731108)

In [28]:
np.amax(fc.third.value)

np.float64(27.29781906731108)

In [33]:
print(ifc3_sc.value[0,0,:,0,0,:,0,0,2].todense())

[[ 0.         27.29781907  0.        ]
 [27.29781907  0.          0.        ]
 [ 0.          0.          0.        ]]


In [ ]:
ifc3_sc = remap_third_force_constants_integer(fc.third, sc)